|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 7:</h2>|<h1>Modern vLLM<h1>|
|<h2>Section:</h2>|<h1>Guided decoding<h1>|
|<h2>Lecture:</h2>|<h1><b>Masking the logits that cannot legally come next<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

# Make the illegal tokens impossible

Asking a model politely for JSON works most of the time, and "most of the
time" is not a schema.

The alternative is to make invalid output unreachable: at every step, set the
logits of every token that cannot legally come next to negative infinity.

In [ ]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

from tests.helpers import json_prefix_state

for s in ['{"a"', '{"a":', '{"a": 1}', '{"a": 1},', '{"a" 1}', '[1, 2', '[1, 2]']:
  print(f'{s!r:<14} {json_prefix_state(s)}')

`prefix` means it could still become valid. `invalid` means no continuation
can rescue it. That second answer is what lets you mask: a token is legal at
this step exactly when appending it leaves you in `prefix` or `valid`.

### The mask, on a tiny vocabulary

In [ ]:
VOCAB = ['{', '}', '[', ']', '"', ':', ',', 'a', 'b', '1', '2', ' ', 'true', 'null']

def legal_mask(prefix):
  return [json_prefix_state(prefix + t) != 'invalid' for t in VOCAB]

for prefix in ['', '{', '{"', '{"a', '{"a"', '{"a":', '{"a": 1']:
  m = legal_mask(prefix)
  allowed = [t for t, ok in zip(VOCAB, m) if ok]
  print(f'{prefix!r:<10} -> {len(allowed):>2}/{len(VOCAB)} legal: {allowed}')

In [ ]:
counts = []
prefix = ''
for step, tokenc in enumerate('{"a": 1, "b": 2}'):
  counts.append(sum(legal_mask(prefix)))
  prefix += tokenc

plt.figure(figsize=(8,3.8))
plt.step(range(len(counts)), counts, where='mid')
plt.xticks(range(len(counts)), list('{"a": 1, "b": 2}'))
plt.ylabel('Legal tokens'); plt.xlabel('Character emitted')
plt.title('How much freedom the grammar leaves, step by step')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

print(f'tightest step: {min(counts)} of {len(VOCAB)} tokens legal')
print('at those steps the model is not choosing at all. The grammar is.')

# The two hard parts, neither of which is the grammar

**Tokenizer alignment.** A real vocabulary is not characters. One token is
`'": '`, another is `'", "'`. A token is legal only if the whole string it
expands to keeps the prefix valid, and the FSM has to be advanced by several
characters at once. Every practical bug in guided decoding lives here.

**Keeping the mask off the critical path.** Building it means asking, for
each of 150,000 tokens, whether appending it is legal.

In [ ]:
import time

VOCAB_SIZE = 151936
per_token_us = 2.0        # a generous estimate for one prefix check

naive_ms = VOCAB_SIZE * per_token_us / 1000
print(f'checking every token, every step: {naive_ms:7.1f} ms')
print(f'a decode step on this model:      {10.0:7.1f} ms')
print(f'\nthe mask would cost {naive_ms/10.0:.0f}x the step it is masking')

So nobody does it that way. The FSM has a **finite** number of states, and a
state determines the mask completely. Precompute one mask per state, cache it,
and a step becomes a lookup and a bitwise operation.

Then build the next state's mask on another thread while the GPU is busy with
the current step, so it is ready before anybody asks.

That is the shape of stage 19: the grammar is the easy part, the tokenizer is
the fiddly part, and the scheduling is the part that makes it free.

    ./vc guide 19